In [1]:
import pandas as pd
from pathlib import Path

processed_dir = Path("../data/processed")

aml_results = pd.read_csv(
    processed_dir / "aml_risk_results.csv"
)

anomaly_results = pd.read_csv(
    processed_dir / "transaction_anomaly_results.csv"
)

financial_results = pd.read_csv(
    processed_dir / "financial_risk_results.csv"
)

print("AML results:", aml_results.shape)
print("Anomaly results:", anomaly_results.shape)
print("Financial risk results:", financial_results.shape)

AML results: (8539, 21)
Anomaly results: (143, 16)
Financial risk results: (6000, 13)


In [2]:
aml_cases = aml_results[
    aml_results["AMLFlag"] == 1
].copy()

aml_cases["CaseSource"] = "AML"
aml_cases["SourceRecordID"] = aml_cases["AMLTransactionID"]
aml_cases["RiskScore"] = aml_cases["AMLScore"]

def aml_risk_level(score):
    if score >= 0.75:
        return "Critical"
    elif score >= 0.50:
        return "High"
    elif score >= 0.21:
        return "Medium"
    return "Low"

aml_cases["RiskLevel"] = aml_cases["RiskScore"].apply(
    aml_risk_level
)

print("AML cases:", len(aml_cases))

print("\nRisk level distribution:")
print(
    aml_cases["RiskLevel"]
    .value_counts()
    .reindex(["Critical", "High", "Medium", "Low"])
    .fillna(0)
    .astype(int)
)

aml_cases[
    [
        "SourceRecordID",
        "RiskScore",
        "RiskLevel",
        "Amount Paid",
        "Payment Currency",
        "Is Laundering",
    ]
].head()

AML cases: 1161

Risk level distribution:
RiskLevel
Critical    418
High        293
Medium      450
Low           0
Name: count, dtype: int64


,SourceRecordID,RiskScore,RiskLevel,Amount Paid,Payment Currency,Is Laundering
5,AML-32,0.920,Critical,16465.55,US Dollar,1
10,AML-42215,0.230,Medium,839.32,US Dollar,0
28,AML-31050,0.250,Medium,1378.93,Euro,1
29,AML-22140,0.615,High,1256.09,US Dollar,0
34,AML-13030,0.425,Medium,6791.69,US Dollar,1


In [3]:
print("Transaction anomaly columns:")
for col in anomaly_results.columns:
    print("-", col)

print("\nFirst 5 rows:")
display(anomaly_results.head())

print("\nData types:")
print(anomaly_results.dtypes)

Transaction anomaly columns:
- TransactionID
- AccountID
- TransactionAmount
- AccountBalance
- TransactionDuration
- LoginAttempts
- AccountAvgTransactionAmount
- AmountDeviationFromAccountAvg
- AmountDeviationZScore
- TransactionToBalanceRatio
- TransactionToAccountAvgRatio
- IsolationAnomalyScore
- IsAnomaly_IF
- DBSCANCluster
- IsAnomaly_DBSCAN
- AnomalyConfidence

First 5 rows:


,TransactionID,AccountID,TransactionAmount,AccountBalance,TransactionDuration,LoginAttempts,AccountAvgTransactionAmount,AmountDeviationFromAccountAvg,AmountDeviationZScore,TransactionToBalanceRatio,TransactionToAccountAvgRatio,IsolationAnomalyScore,IsAnomaly_IF,DBSCANCluster,IsAnomaly_DBSCAN,AnomalyConfidence
0,TX000899,AC00083,1531.31,859.86,62,4,458.896667,1072.413333,1.869265,1.778814,3.329683,0.136035,1,-1,1,High
1,TX000275,AC00454,1176.28,323.69,174,5,494.182500,682.097500,1.448973,3.622779,2.375447,0.128301,1,-1,1,High
2,TX001214,AC00170,1192.20,7816.41,103,5,247.325714,944.874286,2.249026,0.152506,4.800953,0.114565,1,-1,1,High
3,TX002150,AC00110,1250.94,11565.97,107,2,313.471250,937.468750,2.347715,0.108148,3.977915,0.092599,1,-1,1,High
4,TX000148,AC00161,514.95,421.93,142,5,491.475000,23.475000,0.047182,1.217577,1.045637,0.088849,1,-1,1,High



Data types:
TransactionID                     object
AccountID                         object
TransactionAmount                float64
AccountBalance                   float64
TransactionDuration                int64
LoginAttempts                      int64
AccountAvgTransactionAmount      float64
AmountDeviationFromAccountAvg    float64
AmountDeviationZScore            float64
TransactionToBalanceRatio        float64
TransactionToAccountAvgRatio     float64
IsolationAnomalyScore            float64
IsAnomaly_IF                       int64
DBSCANCluster                      int64
IsAnomaly_DBSCAN                   int64
AnomalyConfidence                 object
dtype: object


In [4]:
anomaly_cases = anomaly_results.copy()

anomaly_cases["CaseSource"] = "Anomaly"
anomaly_cases["SourceRecordID"] = anomaly_cases["TransactionID"]


def anomaly_risk_level(row):
    confidence = str(
        row["AnomalyConfidence"]
    ).strip().lower()

    if confidence == "high":
        return "Critical"

    if confidence == "medium":
        return "High"

    return "Medium"


anomaly_cases["RiskLevel"] = anomaly_cases.apply(
    anomaly_risk_level,
    axis=1
)

# Use the saved Isolation Forest anomaly score directly
# as a ranking score.
anomaly_cases["RiskScore"] = (
    anomaly_cases["IsolationAnomalyScore"]
)

print("Anomaly cases:", len(anomaly_cases))

print("\nRisk level distribution:")
print(
    anomaly_cases["RiskLevel"]
    .value_counts()
    .reindex(
        ["Critical", "High", "Medium", "Low"]
    )
    .fillna(0)
    .astype(int)
)

print("\nModel agreement:")
print(
    anomaly_cases[
        [
            "IsAnomaly_IF",
            "IsAnomaly_DBSCAN",
        ]
    ]
    .value_counts()
)

print("\nSample anomaly cases:")
display(
    anomaly_cases[
        [
            "SourceRecordID",
            "AccountID",
            "RiskScore",
            "RiskLevel",
            "AnomalyConfidence",
            "TransactionAmount",
            "IsolationAnomalyScore",
            "IsAnomaly_IF",
            "IsAnomaly_DBSCAN",
        ]
    ].head()
)

Anomaly cases: 143

Risk level distribution:
RiskLevel
Critical    49
High        94
Medium       0
Low          0
Name: count, dtype: int64

Model agreement:
IsAnomaly_IF  IsAnomaly_DBSCAN
1             0                   77
              1                   49
0             1                   17
Name: count, dtype: int64

Sample anomaly cases:


,SourceRecordID,AccountID,RiskScore,RiskLevel,AnomalyConfidence,TransactionAmount,IsolationAnomalyScore,IsAnomaly_IF,IsAnomaly_DBSCAN
0,TX000899,AC00083,0.136035,Critical,High,1531.31,0.136035,1,1
1,TX000275,AC00454,0.128301,Critical,High,1176.28,0.128301,1,1
2,TX001214,AC00170,0.114565,Critical,High,1192.20,0.114565,1,1
3,TX002150,AC00110,0.092599,Critical,High,1250.94,0.092599,1,1
4,TX000148,AC00161,0.088849,Critical,High,514.95,0.088849,1,1


In [5]:
print("Financial risk columns:")
for col in financial_results.columns:
    print("-", col)

print("\nRisk level distribution:")
if "RiskLevel" in financial_results.columns:
    print(financial_results["RiskLevel"].value_counts(dropna=False))

print("\nFirst 5 rows:")
display(financial_results.head())

print("\nData types:")
print(financial_results.dtypes)

Financial risk columns:
- CustomerID
- ActualDefault
- RiskScore
- RiskScorePercent
- RiskLevel
- EarlyWarningFlag
- DelayedPaymentMonths
- MaxPaymentDelay
- RecentPaymentDelay
- CreditUtilization
- PaymentToBillRatio
- FinancialStressIndicatorCount
- RiskReasonSummary

Risk level distribution:
RiskLevel
Low       2460
High      1850
Medium    1690
Name: count, dtype: int64

First 5 rows:


,CustomerID,ActualDefault,RiskScore,RiskScorePercent,RiskLevel,EarlyWarningFlag,DelayedPaymentMonths,MaxPaymentDelay,RecentPaymentDelay,CreditUtilization,PaymentToBillRatio,FinancialStressIndicatorCount,RiskReasonSummary
0,6908,0,0.265464,26.55,Low,0,0,0,0,0.121113,1.003385,0,No major rule-based stress indicator
1,24576,0,0.309089,30.91,Medium,0,0,0,0,0.029660,1.561924,0,No major rule-based stress indicator
2,26767,0,0.381280,38.13,Medium,0,0,0,0,0.988530,0.039392,2,High credit utilization; Low payment-to-bill r...
3,2157,1,0.311544,31.15,Medium,0,0,0,0,0.935412,0.050182,2,High credit utilization; Low payment-to-bill r...
4,3180,0,0.185287,18.53,Low,0,0,0,0,0.021636,1.215185,0,No major rule-based stress indicator



Data types:
CustomerID                         int64
ActualDefault                      int64
RiskScore                        float64
RiskScorePercent                 float64
RiskLevel                         object
EarlyWarningFlag                   int64
DelayedPaymentMonths               int64
MaxPaymentDelay                    int64
RecentPaymentDelay                 int64
CreditUtilization                float64
PaymentToBillRatio               float64
FinancialStressIndicatorCount      int64
RiskReasonSummary                 object
dtype: object


In [6]:
# Keep customers that warrant analyst attention
financial_cases = financial_results[
    financial_results["RiskLevel"].isin(["Medium", "High"])
].copy()

financial_cases["CaseSource"] = "Financial Risk"

# Prefix the customer ID so its provenance remains explicit
financial_cases["SourceRecordID"] = (
    "CUSTOMER-" + financial_cases["CustomerID"].astype(str)
)

# RiskScore and RiskLevel already come from the
# financial-risk engine, so we preserve them unchanged.

print("Financial risk cases:", len(financial_cases))

print("\nRisk level distribution:")
print(
    financial_cases["RiskLevel"]
    .value_counts()
    .reindex(["High", "Medium", "Low"])
    .fillna(0)
    .astype(int)
)

print("\nEarly warning flags:")
print(
    financial_cases["EarlyWarningFlag"]
    .value_counts()
    .sort_index()
)

print("\nSample financial-risk cases:")
display(
    financial_cases[
        [
            "SourceRecordID",
            "CustomerID",
            "RiskScore",
            "RiskLevel",
            "EarlyWarningFlag",
            "DelayedPaymentMonths",
            "CreditUtilization",
            "FinancialStressIndicatorCount",
            "RiskReasonSummary",
        ]
    ].head()
)

Financial risk cases: 3540

Risk level distribution:
RiskLevel
High      1850
Medium    1690
Low          0
Name: count, dtype: int64

Early warning flags:
EarlyWarningFlag
0    1690
1    1850
Name: count, dtype: int64

Sample financial-risk cases:


,SourceRecordID,CustomerID,RiskScore,RiskLevel,EarlyWarningFlag,DelayedPaymentMonths,CreditUtilization,FinancialStressIndicatorCount,RiskReasonSummary
1,CUSTOMER-24576,24576,0.309089,Medium,0,0,0.029660,0,No major rule-based stress indicator
2,CUSTOMER-26767,26767,0.381280,Medium,0,0,0.988530,2,High credit utilization; Low payment-to-bill r...
3,CUSTOMER-2157,2157,0.311544,Medium,0,0,0.935412,2,High credit utilization; Low payment-to-bill r...
5,CUSTOMER-29383,29383,0.471784,High,1,2,0.982943,4,Repeated delayed payments; Severe historical p...
10,CUSTOMER-6909,6909,0.408141,Medium,0,2,0.151063,3,Repeated delayed payments; Severe historical p...


In [7]:
# ---------------------------------------------------------
# Normalize the three engines into one common case schema
# ---------------------------------------------------------

aml_queue = pd.DataFrame({
    "CaseSource": aml_cases["CaseSource"],
    "SourceRecordID": aml_cases["SourceRecordID"],
    "RiskScore": aml_cases["RiskScore"],
    "RiskLevel": aml_cases["RiskLevel"],
    "EntityID": aml_cases["From Account"].astype(str),
    "CaseReason": "AML model flagged transaction",
})

anomaly_queue = pd.DataFrame({
    "CaseSource": anomaly_cases["CaseSource"],
    "SourceRecordID": anomaly_cases["SourceRecordID"],
    "RiskScore": anomaly_cases["RiskScore"],
    "RiskLevel": anomaly_cases["RiskLevel"],
    "EntityID": anomaly_cases["AccountID"].astype(str),
    "CaseReason": (
        anomaly_cases["AnomalyConfidence"].astype(str)
        + " confidence transaction anomaly"
    ),
})

financial_queue = pd.DataFrame({
    "CaseSource": financial_cases["CaseSource"],
    "SourceRecordID": financial_cases["SourceRecordID"],
    "RiskScore": financial_cases["RiskScore"],
    "RiskLevel": financial_cases["RiskLevel"],
    "EntityID": financial_cases["CustomerID"].astype(str),
    "CaseReason": financial_cases["RiskReasonSummary"],
})

# Combine all three sources
unified_cases = pd.concat(
    [
        aml_queue,
        anomaly_queue,
        financial_queue,
    ],
    ignore_index=True,
)

# ---------------------------------------------------------
# Add MonteCore-specific case metadata
# ---------------------------------------------------------

unified_cases.insert(
    0,
    "CaseID",
    [
        f"MC-{i:05d}"
        for i in range(1, len(unified_cases) + 1)
    ],
)

priority_rank = {
    "Critical": 1,
    "High": 2,
    "Medium": 3,
    "Low": 4,
}

unified_cases["PriorityRank"] = (
    unified_cases["RiskLevel"].map(priority_rank)
)

unified_cases["CaseStatus"] = "Open"

# Sort by case priority.
# RiskScore is used only within the source because scores from
# different engines are not directly comparable.
unified_cases = (
    unified_cases
    .sort_values(
        ["PriorityRank", "CaseSource", "RiskScore"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

print("Total unified cases:", len(unified_cases))

print("\nCases by source:")
print(unified_cases["CaseSource"].value_counts())

print("\nCases by priority:")
print(
    unified_cases["RiskLevel"]
    .value_counts()
    .reindex(["Critical", "High", "Medium", "Low"])
    .fillna(0)
    .astype(int)
)

print("\nDuplicate Case IDs:")
print(unified_cases["CaseID"].duplicated().sum())

print("\nMissing priority ranks:")
print(unified_cases["PriorityRank"].isna().sum())

print("\nTop 10 investigation cases:")
display(
    unified_cases[
        [
            "CaseID",
            "CaseSource",
            "SourceRecordID",
            "EntityID",
            "RiskScore",
            "RiskLevel",
            "CaseStatus",
            "CaseReason",
        ]
    ].head(10)
)

Total unified cases: 4844

Cases by source:
CaseSource
Financial Risk    3540
AML               1161
Anomaly            143
Name: count, dtype: int64

Cases by priority:
RiskLevel
Critical     467
High        2237
Medium      2140
Low            0
Name: count, dtype: int64

Duplicate Case IDs:
0

Missing priority ranks:
0

Top 10 investigation cases:


,CaseID,CaseSource,SourceRecordID,EntityID,RiskScore,RiskLevel,CaseStatus,CaseReason
0,MC-00008,AML,AML-51684,8040AE4F0,1.0,Critical,Open,AML model flagged transaction
1,MC-00071,AML,AML-35560,80ED002D0,1.0,Critical,Open,AML model flagged transaction
2,MC-00135,AML,AML-27630,8000FE1F0,1.0,Critical,Open,AML model flagged transaction
3,MC-00137,AML,AML-21145,8019FECD0,1.0,Critical,Open,AML model flagged transaction
4,MC-00185,AML,AML-24988,8000DB3C0,1.0,Critical,Open,AML model flagged transaction
5,MC-00317,AML,AML-46156,805AD2D20,1.0,Critical,Open,AML model flagged transaction
6,MC-00321,AML,AML-7545,8040AE4F0,1.0,Critical,Open,AML model flagged transaction
7,MC-00402,AML,AML-37344,80EBB2100,1.0,Critical,Open,AML model flagged transaction
8,MC-00449,AML,AML-20521,80C0D5B50,1.0,Critical,Open,AML model flagged transaction
9,MC-00493,AML,AML-32124,8021531C0,1.0,Critical,Open,AML model flagged transaction


In [8]:
output_path = processed_dir / "unified_case_queue.csv"

unified_cases.to_csv(
    output_path,
    index=False
)

print("Unified case queue saved.")
print("Location:", output_path)
print("Rows:", len(unified_cases))
print("Columns:", len(unified_cases.columns))

Unified case queue saved.
Location: ..\data\processed\unified_case_queue.csv
Rows: 4844
Columns: 9


In [9]:
queue_check = pd.read_csv(
    processed_dir / "unified_case_queue.csv"
)

print("Export validation")
print("-----------------")

print("Shape:", queue_check.shape)

print(
    "Duplicate Case IDs:",
    queue_check["CaseID"].duplicated().sum()
)

print(
    "Missing values:",
    queue_check.isna().sum().sum()
)

print("\nCases by source:")
print(
    queue_check["CaseSource"].value_counts()
)

print("\nCases by priority:")
print(
    queue_check["RiskLevel"]
    .value_counts()
    .reindex(["Critical", "High", "Medium", "Low"])
    .fillna(0)
    .astype(int)
)

Export validation
-----------------
Shape: (4844, 9)
Duplicate Case IDs: 0
Missing values: 0

Cases by source:
CaseSource
Financial Risk    3540
AML               1161
Anomaly            143
Name: count, dtype: int64

Cases by priority:
RiskLevel
Critical     467
High        2237
Medium      2140
Low            0
Name: count, dtype: int64


In [10]:
import sys
from pathlib import Path

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agents.investigation_agent.schemas import (
    InvestigationCase,
    RiskSignal,
    EvidenceItem,
)

In [11]:
def build_case_from_queue_row(row):
    source = row["CaseSource"]

    risk_signals = []
    evidence = []

    if source == "AML":
        risk_signals.append(
            RiskSignal(
                source="AML Model",
                signal_type="AML Risk",
                score=float(row["RiskScore"]),
                level=row["RiskLevel"],
                description="AML model flagged transaction."
            )
        )

    elif source == "Anomaly":
        risk_signals.append(
            RiskSignal(
                source="Anomaly Engine",
                signal_type="Transaction Anomaly",
                score=float(row["RiskScore"]),
                level=row["RiskLevel"],
                description=row["CaseReason"]
            )
        )

    elif source == "Financial Risk":
        risk_signals.append(
            RiskSignal(
                source="Financial Risk Engine",
                signal_type="Financial Risk",
                score=float(row["RiskScore"]),
                level=row["RiskLevel"],
                description=row["CaseReason"]
            )
        )

    evidence.append(
        EvidenceItem(
            evidence_type="Queue Metadata",
            description=f"Source record: {row['SourceRecordID']}",
            value=row["SourceRecordID"],
            source="Unified Case Queue"
        )
    )

    evidence.append(
        EvidenceItem(
            evidence_type="Queue Metadata",
            description=f"Entity ID: {row['EntityID']}",
            value=row["EntityID"],
            source="Unified Case Queue"
        )
    )

    return InvestigationCase(
        case_id=row["CaseID"],
        customer_id=(
            str(row["EntityID"])
            if source == "Financial Risk"
            else None
        ),
        transaction_id=(
            str(row["SourceRecordID"])
            if source in ["AML", "Anomaly"]
            else None
        ),
        risk_signals=risk_signals,
        evidence=evidence,
        priority=row["RiskLevel"],
        status=row["CaseStatus"],
    )

In [12]:
sample_row = unified_cases.iloc[0]

sample_case = build_case_from_queue_row(
    sample_row
)

print(sample_case)

InvestigationCase(case_id='MC-00008', customer_id=None, transaction_id='AML-51684', risk_signals=[RiskSignal(source='AML Model', signal_type='AML Risk', score=1.0, level='Critical', description='AML model flagged transaction.', metadata={})], evidence=[EvidenceItem(evidence_type='Queue Metadata', description='Source record: AML-51684', value='AML-51684', source='Unified Case Queue', metadata={}), EvidenceItem(evidence_type='Queue Metadata', description='Entity ID: 8040AE4F0', value='8040AE4F0', source='Unified Case Queue', metadata={})], priority='Critical', status='Open')


In [13]:
def attach_source_evidence(
    case,
    queue_row,
    aml_results,
    anomaly_results,
    financial_results,
):
    source = queue_row["CaseSource"]
    source_id = queue_row["SourceRecordID"]

    if source == "AML":

        match = aml_results[
            aml_results["AMLTransactionID"] == source_id
        ]

        if not match.empty:
            row = match.iloc[0]

            aml_evidence = {
                "Timestamp": row["Timestamp"],
                "FromAccount": row["From Account"],
                "ToAccount": row["To Account"],
                "AmountPaid": row["Amount Paid"],
                "PaymentCurrency": row["Payment Currency"],
                "PaymentFormat": row["Payment Format"],
                "CrossCurrency": row["Is Cross Currency"],
                "SameBank": row["Is Same Bank"],
                "SameAccount": row["Is Same Account"],
                "AMLScore": row["AMLScore"],
            }

            for key, value in aml_evidence.items():
                case.evidence.append(
                    EvidenceItem(
                        evidence_type="AML Context",
                        description=f"{key}: {value}",
                        value=value,
                        source="AML Dataset",
                        metadata={"context_key": key},
                    )
                )

    elif source == "Anomaly":

        match = anomaly_results[
            anomaly_results["TransactionID"] == source_id
        ]

        if not match.empty:
            row = match.iloc[0]

            anomaly_evidence = {
                "AccountID": row["AccountID"],
                "TransactionAmount": row["TransactionAmount"],
                "AccountBalance": row["AccountBalance"],
                "TransactionDuration": row["TransactionDuration"],
                "LoginAttempts": row["LoginAttempts"],
                "AmountDeviationZScore": row["AmountDeviationZScore"],
                "TransactionToBalanceRatio": row["TransactionToBalanceRatio"],
                "IsolationAnomalyScore": row["IsolationAnomalyScore"],
                "IsolationForestFlag": row["IsAnomaly_IF"],
                "DBSCANFlag": row["IsAnomaly_DBSCAN"],
                "AnomalyConfidence": row["AnomalyConfidence"],
            }

            for key, value in anomaly_evidence.items():
                case.evidence.append(
                    EvidenceItem(
                        evidence_type="Anomaly Context",
                        description=f"{key}: {value}",
                        value=value,
                        source="Transaction Anomaly Dataset",
                        metadata={"context_key": key},
                    )
                )

    elif source == "Financial Risk":

        customer_id = int(queue_row["EntityID"])

        match = financial_results[
            financial_results["CustomerID"] == customer_id
        ]

        if not match.empty:
            row = match.iloc[0]

            financial_evidence = {
                "CustomerID": row["CustomerID"],
                "RiskScore": row["RiskScore"],
                "DelayedPaymentMonths": row["DelayedPaymentMonths"],
                "MaxPaymentDelay": row["MaxPaymentDelay"],
                "RecentPaymentDelay": row["RecentPaymentDelay"],
                "CreditUtilization": row["CreditUtilization"],
                "PaymentToBillRatio": row["PaymentToBillRatio"],
                "FinancialStressIndicatorCount":
                    row["FinancialStressIndicatorCount"],
                "RiskReasonSummary": row["RiskReasonSummary"],
            }

            for key, value in financial_evidence.items():
                case.evidence.append(
                    EvidenceItem(
                        evidence_type="Financial Risk Context",
                        description=f"{key}: {value}",
                        value=value,
                        source="Financial Risk Dataset",
                        metadata={"context_key": key},
                    )
                )

    return case

In [14]:
sample_case = attach_source_evidence(
    sample_case,
    sample_row,
    aml_results,
    anomaly_results,
    financial_results,
)

print("Case:", sample_case.case_id)
print("Source:", sample_row["CaseSource"])
print("Evidence count:", len(sample_case.evidence))

print("\nEvidence:")
for item in sample_case.evidence:
    print(
        f"- [{item.evidence_type}] "
        f"{item.description}"
    )

Case: MC-00008
Source: AML
Evidence count: 12

Evidence:
- [Queue Metadata] Source record: AML-51684
- [Queue Metadata] Entity ID: 8040AE4F0
- [AML Context] Timestamp: 2022-09-11 13:03:00
- [AML Context] FromAccount: 8040AE4F0
- [AML Context] ToAccount: 8001694F0
- [AML Context] AmountPaid: 9411.91
- [AML Context] PaymentCurrency: Euro
- [AML Context] PaymentFormat: ACH
- [AML Context] CrossCurrency: 0
- [AML Context] SameBank: 0
- [AML Context] SameAccount: 0
- [AML Context] AMLScore: 1.0


In [15]:
anomaly_sample_row = unified_cases[
    unified_cases["CaseSource"] == "Anomaly"
].iloc[0]

anomaly_sample_case = build_case_from_queue_row(
    anomaly_sample_row
)

anomaly_sample_case = attach_source_evidence(
    anomaly_sample_case,
    anomaly_sample_row,
    aml_results,
    anomaly_results,
    financial_results,
)

print("Case:", anomaly_sample_case.case_id)
print("Source:", anomaly_sample_row["CaseSource"])
print("Evidence count:", len(anomaly_sample_case.evidence))

print("\nEvidence:")
for item in anomaly_sample_case.evidence:
    print(
        f"- [{item.evidence_type}] "
        f"{item.description}"
    )

Case: MC-01162
Source: Anomaly
Evidence count: 13

Evidence:
- [Queue Metadata] Source record: TX000899
- [Queue Metadata] Entity ID: AC00083
- [Anomaly Context] AccountID: AC00083
- [Anomaly Context] TransactionAmount: 1531.31
- [Anomaly Context] AccountBalance: 859.86
- [Anomaly Context] TransactionDuration: 62
- [Anomaly Context] LoginAttempts: 4
- [Anomaly Context] AmountDeviationZScore: 1.8692654024638444
- [Anomaly Context] TransactionToBalanceRatio: 1.7788142090467671
- [Anomaly Context] IsolationAnomalyScore: 0.1360353583308299
- [Anomaly Context] IsolationForestFlag: 1
- [Anomaly Context] DBSCANFlag: 1
- [Anomaly Context] AnomalyConfidence: High


In [16]:
financial_sample_row = unified_cases[
    unified_cases["CaseSource"] == "Financial Risk"
].iloc[0]

financial_sample_case = build_case_from_queue_row(
    financial_sample_row
)

financial_sample_case = attach_source_evidence(
    financial_sample_case,
    financial_sample_row,
    aml_results,
    anomaly_results,
    financial_results,
)

print("Case:", financial_sample_case.case_id)
print("Source:", financial_sample_row["CaseSource"])
print("Evidence count:", len(financial_sample_case.evidence))

print("\nEvidence:")
for item in financial_sample_case.evidence:
    print(
        f"- [{item.evidence_type}] "
        f"{item.description}"
    )

Case: MC-03612
Source: Financial Risk
Evidence count: 11

Evidence:
- [Queue Metadata] Source record: CUSTOMER-27537
- [Queue Metadata] Entity ID: 27537
- [Financial Risk Context] CustomerID: 27537
- [Financial Risk Context] RiskScore: 0.967043465734444
- [Financial Risk Context] DelayedPaymentMonths: 6
- [Financial Risk Context] MaxPaymentDelay: 3
- [Financial Risk Context] RecentPaymentDelay: 3
- [Financial Risk Context] CreditUtilization: 0.009948
- [Financial Risk Context] PaymentToBillRatio: 0.0
- [Financial Risk Context] FinancialStressIndicatorCount: 4
- [Financial Risk Context] RiskReasonSummary: Repeated delayed payments; Severe historical payment delay; Recent payment delinquency; Low payment-to-bill ratio; Repayment behaviour deteriorating; Multiple financial stress indicators


In [17]:
import sys
from pathlib import Path

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agents.investigation_agent.orchestrator import (
    run_agentic_investigation
)

agent_result = run_agentic_investigation(
    anomaly_sample_case
)

print("Deterministic result available:",
      agent_result.deterministic_result is not None)

print("Primary Mistral summary available:",
      agent_result.primary_summary is not None)

print("Llama review available:",
      agent_result.review_output is not None)

print("\nPrimary error:")
print(agent_result.primary_error)

print("\nReview error:")
print(agent_result.review_error)

Deterministic result available: True
Primary Mistral summary available: True
Llama review available: True

Primary error:
None

Review error:
None


In [18]:
print("=== CASE ===")
print("Case ID:", anomaly_sample_case.case_id)
print("Priority:", anomaly_sample_case.priority)

print("\n=== DETERMINISTIC SUMMARY ===")
print(agent_result.deterministic_result.summary)

print("\n=== MISTRAL INVESTIGATION ===")
print(agent_result.primary_summary)

print("\n=== LLAMA SECOND-LOOK REVIEW ===")
print(agent_result.review_output)

=== CASE ===
Case ID: MC-01162
Priority: Critical

=== DETERMINISTIC SUMMARY ===
Case MC-01162 is classified as Critical priority based on 1 risk signals and 13 supporting evidence items.

=== MISTRAL INVESTIGATION ===
## Case Summary
The case involves a high-priority anomaly detected on transaction TX000899 linked to account AC00083. The anomaly is marked as critical due to substantial deviations in transaction behavior and account activity.

## Key Risk Indicators
- **Critical Transaction Anomaly**: High confidence score of 0.136.
- **Overdrawn Transaction**: Amount of 1,531.31 exceeded the account balance of 859.86.
- **Multiple Login Attempts**: 4 login attempts recorded during the transaction period.
- **Significant Amount Deviation**: Z-score of 1.869 indicating a substantial deviation from the norm.
- **High Transaction-to-Balance Ratio**: Ratio of 1.778 indicates the transaction amount is nearly double the account balance.
- **Anomaly Detection Flags**: Both IsolationForestFlag

In [19]:
from agents.investigation_agent.orchestrator import (
    run_agentic_investigation
)

agent_result = run_agentic_investigation(
    anomaly_sample_case
)

print(
    "Deterministic result available:",
    agent_result.deterministic_result is not None
)

print(
    "Primary Mistral summary available:",
    agent_result.primary_summary is not None
)

print(
    "Llama review available:",
    agent_result.review_output is not None
)

print("\nPrimary error:")
print(agent_result.primary_error)

print("\nReview error:")
print(agent_result.review_error)

Deterministic result available: True
Primary Mistral summary available: True
Llama review available: True

Primary error:
None

Review error:
None


In [21]:
import sys
import importlib
from pathlib import Path

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Clear cached agents modules so Python sees the newly created package
for module_name in list(sys.modules):
    if module_name == "agents" or module_name.startswith("agents.case_engine"):
        del sys.modules[module_name]

importlib.invalidate_caches()

print("Project root:", project_root)
print("Project root in sys.path:", str(project_root) in sys.path)
print(
    "case_engine exists:",
    (project_root / "agents" / "case_engine").exists()
)
print(
    "case_manager.py exists:",
    (project_root / "agents" / "case_engine" / "case_manager.py").exists()
)

Project root: C:\Users\91869\Documents\projects\montecore-finance
Project root in sys.path: True
case_engine exists: False
case_manager.py exists: False


In [24]:
import importlib
import agents.case_agent.case_manager as case_manager

importlib.reload(case_manager)

print("Processed directory:")
print(case_manager.DEFAULT_PROCESSED_DIR)

print(
    "Directory exists:",
    case_manager.DEFAULT_PROCESSED_DIR.exists()
)

rebuilt_case = case_manager.build_investigation_case(
    "MC-01162"
)

print("\nCase ID:", rebuilt_case.case_id)
print("Priority:", rebuilt_case.priority)
print("Transaction ID:", rebuilt_case.transaction_id)
print("Customer ID:", rebuilt_case.customer_id)
print("Risk signal count:", len(rebuilt_case.risk_signals))
print("Evidence count:", len(rebuilt_case.evidence))

Processed directory:
C:\Users\91869\Documents\projects\montecore-finance\data\processed
Directory exists: True

Case ID: MC-01162
Priority: Critical
Transaction ID: TX000899
Customer ID: None
Risk signal count: 1
Evidence count: 13
